# SBA Data Cleaning Notebook

In [1]:
import pandas as pd

## Exploring data structure

In [2]:
sba_data = pd.read_csv('../data/raw/sba_data_2010-19.csv', low_memory=False)

### Ensuring reasonable distribution of states and Paid in Full v. Default (CHGOFF)

In [3]:
print(sba_data.shape)
print(sba_data["loanstatus"].value_counts())
print(sba_data["borrstate"].value_counts())

(545751, 43)
loanstatus
PIF       389005
CANCLD     66399
EXEMPT     56869
CHGOFF     33418
COMMIT        60
Name: count, dtype: int64
borrstate
CA    68070
TX    41841
NY    35267
OH    30880
FL    26345
MI    21267
MA    19616
IL    17994
PA    17552
MN    15408
NJ    15248
GA    14915
WI    14541
WA    14523
CO    13396
IN    12011
AZ    11079
UT    10922
NC    10763
MO     9964
VA     8369
OR     7860
MD     7802
CT     6146
ID     5222
NH     5050
KY     4984
TN     4880
OK     4799
MS     4718
PR     4659
KS     4348
IA     4342
LA     4332
SC     4166
NV     4156
AL     3867
NE     3777
RI     3345
ME     3319
HI     3082
AR     3051
MT     2984
NM     2596
VT     2327
ND     1870
WV     1489
DE     1473
SD     1388
AK     1190
WY     1081
DC      903
GU      449
VI      104
MP       16
FM        3
MH        1
AS        1
Name: count, dtype: int64


In [4]:
sba_data.shape

(545751, 43)

In [5]:
print(sba_data.columns.tolist())

['asofdate', 'program', 'l2locid', 'borrname', 'borrstreet', 'borrcity', 'borrstate', 'borrzip', 'bankname', 'bankfdicnumber', 'bankncuanumber', 'bankstreet', 'bankcity', 'bankstate', 'bankzip', 'grossapproval', 'sbaguaranteedapproval', 'approvaldate', 'approvalfiscalyear', 'firstdisbursementdate', 'processingmethod', 'subprogram', 'initialinterestrate', 'fixedorvariableinterestind', 'terminmonths', 'naicscode', 'naicsdescription', 'franchisecode', 'franchisename', 'projectcounty', 'projectstate', 'sbadistrictoffice', 'congressionaldistrict', 'businesstype', 'businessage', 'loanstatus', 'paidinfulldate', 'chargeoffdate', 'grosschargeoffamount', 'revolverstatus', 'jobssupported', 'collateralind', 'soldsecmrktind']


### Dropping columns that are not necessary for analysis

In [6]:
drop_cols = ['borrname', 'borrstreet', 'borrzip', 'bankname', 'bankfdicnumber', 'bankncuanumber',
              'bankstreet', 'bankcity', 'bankzip','processingmethod', 'subprogram',
               'franchisecode', 'franchisename', 'firstdisbursementdate','sbadistrictoffice', 
               'congressionaldistrict','paidinfulldate', 'chargeoffdate','grosschargeoffamount', 'collateralind',
               'asofdate', 'program', 'l2locid', 'soldsecmrktind', 'projectcounty']
sba_data_cleaned = sba_data.drop(columns=drop_cols)

In [7]:
sba_data_cleaned.shape

(545751, 18)

### Feature engineering new analysis fields

In [8]:
# SBA guarantee percentage
sba_data_cleaned["guarantee_pct"] = sba_data_cleaned["sbaguaranteedapproval"] / sba_data_cleaned["grossapproval"]

# NAICS sector (first 2 digits)
sba_data_cleaned["naics_sector"] = sba_data_cleaned["naicscode"].astype(str).str[:2]

# Project in same state as borrower
sba_data_cleaned["projectinstate"] = (sba_data_cleaned["borrstate"] == sba_data_cleaned["projectstate"]).astype(int)

# Bank in same state as borrower
sba_data_cleaned["bankinstate"] = (sba_data_cleaned["borrstate"] == sba_data_cleaned["bankstate"]).astype(int)

### Keeping only paid in full or defaulted loans

In [9]:
# checking what statuses exits
print(sba_data_cleaned["loanstatus"].value_counts())

# keeping only resolved statuses (PIF and CHGOFF)
sba_data_cleaned = sba_data_cleaned[sba_data_cleaned["loanstatus"].isin(["PIF", "CHGOFF"])].copy()

# making target (default) binary
sba_data_cleaned["default"] = (sba_data_cleaned["loanstatus"] == "CHGOFF").astype(int)

print(f"Remaining rows: {len(sba_data_cleaned)}")
print(f"Default rate: {sba_data_cleaned['default'].mean():.4f}")

loanstatus
PIF       389005
CANCLD     66399
EXEMPT     56869
CHGOFF     33418
COMMIT        60
Name: count, dtype: int64
Remaining rows: 422423
Default rate: 0.0791


### Saving lookup variable later for potential further analysis

In [10]:
sector_lookup = (sba_data_cleaned[["naics_sector", "naicsdescription"]]
                 .drop_duplicates()
                 .sort_values("naics_sector"))
print(sector_lookup.head(20))

       naics_sector                          naicsdescription
15551            11                           Cattle Feedlots
44179            11     Crop Harvesting, Primarily by Machine
25857            11  Dual-Purpose Cattle Ranching and Farming
2936             11             Other Noncitrus Fruit Farming
2059             11          Beef Cattle Ranching and Farming
8265             11                              Corn Farming
6716             11                        Poultry Hatcheries
8917             11                          Tree Nut Farming
42953            11                            Cotton Ginning
49690            11                      Hunting and Trapping
4421             11         Horse and Other Equine Production
10583            11         Berry (except Strawberry) Farming
15148            11                  Other Animal Aquaculture
5856             11                           Grape Vineyards
11358            11               All Other Animal Production
113876  

In [11]:
sba_data_cleaned = sba_data_cleaned.drop(columns=["bankstate", "projectstate", "borrcity", "loanstatus", "approvaldate"])

### Dataset structure checks

In [12]:
# finding dtypes
print(sba_data_cleaned.dtypes)

borrstate                      object
grossapproval                   int64
sbaguaranteedapproval         float64
approvalfiscalyear              int64
initialinterestrate           float64
fixedorvariableinterestind     object
terminmonths                    int64
naicscode                     float64
naicsdescription               object
businesstype                   object
businessage                    object
revolverstatus                  int64
jobssupported                 float64
guarantee_pct                 float64
naics_sector                   object
projectinstate                  int64
bankinstate                     int64
default                         int64
dtype: object


In [13]:
# Check for nulls
print(sba_data_cleaned.isnull().sum())

borrstate                          0
grossapproval                      0
sbaguaranteedapproval              0
approvalfiscalyear                 0
initialinterestrate                0
fixedorvariableinterestind         0
terminmonths                       0
naicscode                          2
naicsdescription                 432
businesstype                       0
businessage                   352237
revolverstatus                     0
jobssupported                      0
guarantee_pct                      0
naics_sector                       0
projectinstate                     0
bankinstate                        0
default                            0
dtype: int64


In [14]:
# dropping businessage (mostly missing)
sba_data_cleaned = sba_data_cleaned.drop(columns=["businessage"])

### Checking specific variable values

In [15]:
# - revolverstatus: what values?
print(sba_data_cleaned["revolverstatus"].value_counts())

# - fixedorvariableinterestind: what values?
print(sba_data_cleaned["fixedorvariableinterestind"].value_counts())

# - businesstype: what values?
print(sba_data_cleaned["businesstype"].value_counts())

revolverstatus
0    282218
1    140205
Name: count, dtype: int64
fixedorvariableinterestind
V    330957
F     91466
Name: count, dtype: int64
businesstype
CORPORATION    367744
INDIVIDUAL      46932
PARTNERSHIP      7727
                   20
Name: count, dtype: int64


### Encoding

In [16]:
# Encoding variable rate
sba_data_cleaned["variable_rate"] = (sba_data_cleaned["fixedorvariableinterestind"] == "V").astype(int)
sba_data_cleaned = sba_data_cleaned.drop(columns=["fixedorvariableinterestind"])

In [17]:
sba_data_cleaned.columns

Index(['borrstate', 'grossapproval', 'sbaguaranteedapproval',
       'approvalfiscalyear', 'initialinterestrate', 'terminmonths',
       'naicscode', 'naicsdescription', 'businesstype', 'revolverstatus',
       'jobssupported', 'guarantee_pct', 'naics_sector', 'projectinstate',
       'bankinstate', 'default', 'variable_rate'],
      dtype='object')

In [18]:
# dropping code (no longer needed)
sba_data_cleaned = sba_data_cleaned.drop(columns=["naicscode"])

### Final Checks

In [19]:
print(f"Shape: {sba_data_cleaned.shape}")
print(f"\nColumns: {sba_data_cleaned.columns.tolist()}")
print(f"\nDtypes:\n{sba_data_cleaned.dtypes}")
print(f"\nNulls:\n{sba_data_cleaned.isnull().sum()}")
print(f"\nDefault rate: {sba_data_cleaned['default'].mean()*100:.2f}%")

Shape: (422423, 16)

Columns: ['borrstate', 'grossapproval', 'sbaguaranteedapproval', 'approvalfiscalyear', 'initialinterestrate', 'terminmonths', 'naicsdescription', 'businesstype', 'revolverstatus', 'jobssupported', 'guarantee_pct', 'naics_sector', 'projectinstate', 'bankinstate', 'default', 'variable_rate']

Dtypes:
borrstate                 object
grossapproval              int64
sbaguaranteedapproval    float64
approvalfiscalyear         int64
initialinterestrate      float64
terminmonths               int64
naicsdescription          object
businesstype              object
revolverstatus             int64
jobssupported            float64
guarantee_pct            float64
naics_sector              object
projectinstate             int64
bankinstate                int64
default                    int64
variable_rate              int64
dtype: object

Nulls:
borrstate                  0
grossapproval              0
sbaguaranteedapproval      0
approvalfiscalyear         0
initialintere

### Saving to cleaned csv

In [20]:
sba_data_cleaned.to_csv('../data/clean/sba_data_cleaned.csv', index=False)